<h1> Task 2: Radar Imaging</h1>


<h3> 2.0 3D Imaging using 2D Antenna Array </h3>

This task includes two parts. Our goal is to implement radar signal processing for 3D imaging
using a 2D antenna array. You will implement two algorithms we taught in the class. 

<h5> 2.1 Algorithm 1 - Conventional Beam Forming </h5>

In this part, we will need to image the scene using Algorithm 1 we taught in class for a 2D antenna array. You will generate the 3D radar heatmap by estimating the
reflected signal power from every azimuth angles $\phi$ - elevation angles $\theta$ pair. Every azimuth
angle and elevation angle, along with the range will pinpoint a 3D voxel in the spherical
coordinates. To speed up the alogrithm, you can crop the range bins to be from 100 to 110 instead of using all 512 range bins.
The code for this task should be written in the functions *beamform_2d* defined below. We provide you
the code to load the radar data in the right format and to plot the 3D heatmap in this
file.

As results, you need to include the following in your report.
1. Generate a 3D heatmap for **all data given** from beamforming for azimuth angle $\phi$ between 60 and 130 degrees with a resolution
of 1 degree, and elevation angle $\theta$ between 70 and 110 degrees with a resolution of 1 degrees. Include the top view, side view, and
front view projections of the 3D heatmap in the spherical coordinates. 


Write your algorithm below.

In [ ]:
################# Change the values based on how much of the azimuth angles you want to see and the resolution ##################
# TODO: write beamforming 2D code
# Define field of view in degrees that you want to process in theta, phi and range bins
def beamform_2d(beat_freq_data, phi_s, phi_e, phi_res, theta_s, theta_e, theta_res, x_idx, z_idx, r_idxs, radar_params):
    """
    Performs 2D beamforming along the azimuth (horizontal) dimension, this results in a bird eye view image.
    - beat_freq_data: beat data AKA the range FFT (size: num_x_stps * num_z_stps * num TX * num RX, num ADC samples)
    - phi_s: first azimuth angle that you want to start computing 
    - phi_e: last azimuth angle that you want to compute 
    - phi_res: resolution of the azimuth angles you want to compute
    - theta_s: first elevation angle that you want to start computing 
    - theta_e: last elevation angle that you want to compute 
    - theta_res: resolution of the elevation angles you want to compute
    - x_locs: x coordinate of antenna locations
    - z_locs: z coordinate of antenna locations
    - r_idx: range bins to calculate 
    - radar_parms: radar_params if needed 

    Returns:
    - sph_pwr: beamformed result (size: n_phi, n_theta, n_range)
    - phi: array of azimuth angles 
    - theta: array of elevation angles 
    """
    sph_pwr = 0
    phi = 0
    theta = 0
    return sph_pwr, phi, theta 

In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import os
import scipy
import scipy.io as sio
import time
# import open3d as o3d
import utils 

Load data.

In [ ]:
# Path to the data
data_path = r"data/data_003_.mat"

# loading data that is given
"""
    raw_data: is the raw radar data (after mixing) of size (num_x_stp * num_rx, num_z_stp, adc_samples)
    radar_params: is a dictionary with radar and position parameters: 'sample_rate', 'num_samples', 'slope', 'lm'(lambda), 'num_x_stp', 'num_z_stp', 'num_tx', 'num_rx', 'adc_samples' 
"""
radar_params, raw_data = utils.load_raw_data(data_path)

Defining antenna positions. 

In [ ]:
x_ant_pos, z_pos, x_radar = utils.get_ant_pos_2d(radar_params['num_x_stp'], radar_params['num_z_stp'], radar_params['num_rx']) # this returns x position of rx, z positions of rx and tx, and x position of tx

Process raw data for BF algorthim (use all samples now).

In [ ]:
# TODO: reshape raw_data and process beat frequency
beat_freq = 0

Define the angles and range bins to run, and run the BF algo. For reference, our implementation takes around 1 minute with the full image resolution on data_003_. Depending on how you write your code it might take longer (hint: Python (specifically NumPy) is high optimized for vectorized code. So it is generally much faster to use array multiplication than doing explicit for loops).

In [ ]:
# define the azimuth angles (horizontal FOV) that we want to look at 
r_idxs = np.arange(100, 110, 1)
phi_s, phi_e = 60, 130 
phi_res = 1
theta_s, theta_e = 70, 110 
theta_res = 1

# Run your algorithm here
bf_output1, phi, theta = beamform_2d(beat_freq, phi_s,phi_e,phi_res,theta_s,theta_e,theta_res,x_ant_pos,z_pos,r_idxs,radar_params)

Here you should plot the outputs from beamforming.

In [ ]:
# Plot the output from 2D Beamforming (You can change this as you see fit)
fig = plt.figure(figsize=(20, 7))

to_plot = np.sum(abs(bf_output1),axis=-1)
to_plot = to_plot/np.max(np.reshape(to_plot,(1,-1))) 
to_plot = to_plot**2
ax0 = fig.add_subplot(131)
utils.plot_2d_heatmap(ax0, to_plot, phi, theta, vmin=0, vmax=0.1)
ax0.title.set_text('Front View')

to_plot = np.sum(abs(bf_output1),axis=1)
to_plot = to_plot/np.max(np.reshape(to_plot,(1,-1))) 
to_plot = to_plot**2
ax1 = fig.add_subplot(132,projection = 'polar')
utils.plot_2d_heatmap(ax1, to_plot, phi, r_idxs, vmin=0, vmax=0.1)
ax1.title.set_text('Bird Eye View (Top View)')

to_plot = np.sum(abs(bf_output1),axis=0)
to_plot = to_plot/np.max(np.reshape(to_plot,(1,-1))) 
to_plot = to_plot**2
ax0 = fig.add_subplot(133)
utils.plot_2d_heatmap(ax0, to_plot, theta, r_idxs, vmin=0, vmax=0.1)
ax0.title.set_text('Side View')

plt.tight_layout()
plt.show()

<h1> Task 2: Radar Imaging Part 2</h1>


<h5> 2.2 Algorithm 3 - Matched Filter </h5>

In this part, we will need to image the scene using Algorithm 3 we saw in class for a 2D antenna array. The algorithm will be described below. You will generate the 3D radar heatmap by estimating
the reflected signal power from every voxel in the 3D cartesian coordinates (x,y,z). Here, x is horizontal or azimuth, y is depth, and z is vertical or elevation.
The code for this task should be written in the function matched_filter_time_2d. We provide you
the code to load the radar data in the right format and to plot the heatmap in this
file. Since we are only plotting a small slice in depth you only need to provide images of the front plot (code is provided).

<h5>Matched Filter</h5>
In matched_filter_time_2d, write the corresponding function following the formulation explained in the lecture as Algorithm 3 (the main difference, similar to the Tutorial is that the phase is negated in relation to the lecture):

$$
P(x,y,z) = \sum_{m=1}^{l_N} \sum_{k=1}^{N} \sum_{t=1}^{T}
s_{m,k}(t)\, e^{-j 4\pi\left(\frac{p t}{c} + \frac{1}{\lambda}\right)
\sqrt{(x - m s)^2 + (y - k s)^2 + (z - z_c)^2}}
$$

Here $s_{m,k}$ is the raw data signal from $ant_{m,k}$ and $p$ is the slope. We can assume in this lab that the transmitter and receiver are colocated, so you do not need to account for the difference in location. However, generally in practice we usually account for this difference in the round trip distance calculation to account for the different locations. 

As results, you need to include the following in the turned in files: 
1. Generate the heatmap for front views 
in the cartesian coordinates. The space of interest is indicated in Data Details the data folder. The spatial resolution
should be 0.025 for x and z and 0.05 resolution for y. 

2. Compare the heatmaps with the corresponding one from Task 3a, and comment on
the differences.

Notes: 
- There is a **0.15m circuit delay** that needs to be added to the computed distance. 
- Running matched filter code in Python can be slow, try to optimize with Python. One thing you can do, is *subsample the adc_samples* (eg. take one time domain signal every 32 indexes) and *divide the sampling rate by that same value*. This essentially changes maximum *range* of the signal without affecting the range resolution. But since the objects imaged are around 4-5m away, this does not work super well since it lowers the max range. Another option is to just take a subset of ADC samples, however, this severely impacts the range resolution. We have implemented code that already does this, and you should assume the ADC samples are cropped between time index 100 and 130 (see matched filter function).
- And remember to check parameter clim to get reasonable heatmaps. 


Implement your time matched filter algortihm below. If you would like to speed up processing time you can try to optimize code --> (eg. Numba, multiprocessing,), additionlly for debugging we recommend processing lower resolution.

Additionally, you may move away from a Jupyter Notebook for this task only if you find that optmizing the code requires this. However, please attach the plots in a PDF in addition to plotting the output in your code if you follow this path.

In [ ]:
######## 2D Imaging in Cartesian #############
# TODO: complete this function
def matched_filter_time_2d(raw_data_2d, x_cells, z_cells, y_cells, x_radar_rx, z_radar_rx):
    """
    Computes the 2D matched filter result for raw data, this is done along the X, Y dimensions (results in a birds eye view image).

    Paramters:
    - raw_data_2d: raw ADC data (size: num Tx * num Rx * horizontal steps * vertical steps, num ADC samples)
    - x_cells: discrete locations in X (horizontal) that you want to compute the power of
    - z_cells: discrete locations in Z (vertical) that you want to compute the power of
    - y_cells: depth slice we want to calculate
    - x_radar_rx: horizontal positions of the receivers 
    - z_radar_rx: vertical positions of the receivers 

    Returns:
    - MF_output: matched filter output of size (plotting code assume it is of size: (len(x_cells), len(y_cells), len(z_cells))
    """
    c = 2.9979
    fc_start =  773.704
    chirpSlope = 70.295e12
    adcSampleRate = 10e6
    As_sci = (chirpSlope/1e8)

    # to speed up processing just look at this subset of samples
    adc_samples_cropped = np.arange(100,130) # you can change this if you would like
    raw_data_2d = raw_data_2d[:,adc_samples_cropped] # crop raw data to adc_samples subset
    adc_samples_mf = raw_data_2d.shape[1] # number adc samples after cropping aka 30

    MF_output = np.zeros((len(x_cells),len(y_cells),len(z_cells)),dtype=complex)
    return MF_output 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import os
import scipy
import scipy.io as sio
import time
import open3d as o3d
import utils

Load data.

In [ ]:
# TODO: Put the *path* to the project folder
data_path = r"data/data_003_.mat"

# loading data that is given
"""
    raw_data: is the raw radar data (after mixing) of size (num_x_stp x num_rx, num_z_stp, adc_samples)
    radar_params: is a dictionary with radar and position parameters: 'sample_rate', 'num_samples', 'slope', 'lm'(lambda), 'num_x_stp', 'num_z_stp', 'num_tx', 'num_rx', 'adc_samples' 
"""
radar_params, raw_data = utils.load_raw_data(data_path)

Defining antenna positions. This returns x position of rx, z positions of rx, and x position of the radar. Since the radar has 4 receivers and 1 transmitter, technically z_ant_pos is also the z positions of the transmitter and x_radar is the x positions of the transmitter (offset by the distance from the transmitter to the first receiver which is aroun 0.005 + 3 $\lambda$ /2). However, in this lab we can ignore the distance between transmitter and receiver when calculating the matched filter. 

In [ ]:
x_ant_pos, z_ant_pos, x_radar = utils.get_ant_pos_2d(radar_params['num_x_stp'], radar_params['num_z_stp'], radar_params['num_rx']) # 

Process raw data for MF algorthim (use all samples now).

In [ ]:
# TODO: reshape raw_data for MF function
X = 0 

Define the x,y,z voxels to calculate and the radars transmitter and receiver positions in x and z. Then run your algorithm. We recommend processing at lower resolution (eg. 0.05) or smaller portions of space for debugging. For reference, our code takes around 3 minutes to run the full image with full resolution for data_003_, 6 minutes for data_001_, and 3 minutes for data_010_.

In [ ]:
# TODO: This is values for data_003_ but you can change it for other values
# You can also lower the resolution to speed up processing 
# These bounds are for data_003_
x_cells = np.arange(-0.12,0.46,0.025) # horizontal
y_cells = np.arange(4.3,4.4,0.05) # depth
z_cells = np.arange(-0.40,0.25,0.025) # vertical

# transmitters x position is one every lambda (repeat by four to match size of x_radar_rx) and shift by the physical offset on the board
x_radar_rx = x_ant_pos # size 320 x 1
z_radar_rx = z_ant_pos # size 1 x 369

# Run matched filter algorithm
MF_output = matched_filter_time_2d(X, x_cells, y_cells, z_cells, x_radar_rx, z_radar_rx)

Plot your outputs below.

In [ ]:
# Plot the output from 2D Matched Filtering (You can change this as you want)
fig = plt.figure(figsize=(7, 7))

to_plot = np.sum(abs(MF_output),axis=1)
to_plot = to_plot/np.max(np.reshape(to_plot,(1,-1))) 
to_plot = to_plot**2
ax1 = fig.add_subplot(111)
utils.plot_2d_heatmap(ax1, to_plot, x_cells, z_cells, vmin=0, vmax=0.1)
ax1.title.set_text('Front View')

plt.tight_layout()
plt.show()